# Hands-On Workshop: Building a Data Lake
**Data Ecosystems and Governance in Organizations (DEGO 2606)**  
Nova School of Business and Economics — MSc Business Analytics

---

This notebook walks you through building a mini data lake from scratch using real e-commerce data. You will ingest raw data, clean it, store it in Delta Lake format, run SQL queries, and explore time travel — the same workflow used in production data engineering teams.

**How to run:** execute cells one by one using `Shift + Enter` or the ▶ button. Do not skip cells and do not run them out of order.

**The dataset:** ~541,000 retail transactions from a UK-based online gift shop (2010–2011), sourced from the UCI Machine Learning Repository. It is intentionally messy — missing customer IDs, cancelled orders, negative quantities. That is the point.

---
## Setup

Databricks does not ship with an Excel reader by default, so we install `openpyxl` first. The `%pip` command installs it directly into your cloud environment — nothing is downloaded to your laptop. After the install, we need to restart Python so the new package is recognised. **Run cells 1 and 2, then continue from cell 3.**

In [0]:
# Install the Excel reader library.
# The --quiet flag suppresses the installation log to keep output clean.
%pip install openpyxl --quiet


In [0]:
# Restart the Python interpreter so the newly installed package is available.
# The notebook will briefly show a 'kernel restarting' message — this is normal.
dbutils.library.restartPython()


---
## Part 1 — Raw Zone (Bronze Layer)

In a real data lake, the raw zone holds data exactly as it arrived — no cleaning, no transformation. Think of it as the archive: you always keep the original so you can reprocess if needed.

Here we load the dataset directly from a URL into a pandas DataFrame, fix a type issue, then hand it off to Spark. The reason we go through pandas first is that `spark.read` cannot read `.xlsx` files natively — it is designed for CSV, Parquet, Delta, and similar formats. pandas can handle Excel, so we use it as a bridge.

**Why does the type-fixing loop exist?**  
The `InvoiceNo` column contains mostly numbers but also strings like `C536379` (the `C` prefix marks a cancellation). pandas reads this as a mixed-type column, which causes an error when Spark tries to convert it. Casting all text columns to `string` before conversion solves this cleanly.

In [0]:
import pandas as pd

DROPBOX_URL = (
    "https://www.dropbox.com/scl/fi/eiuygthq63p18by82mmn1/"
    "Online-Retail.xlsx"
    "?rlkey=1w97yb3oiddi6rxaqa0pksttf&st=bpkflqw7&dl=1"
)

# pandas reads the Excel file directly from the URL.
# openpyxl is the engine that handles the .xlsx format.
print("Loading dataset...")
pandas_df = pd.read_excel(DROPBOX_URL, engine="openpyxl")

# The dataset has mixed-type columns (text + numbers in the same column).
# We cast every text column to a clean string type before passing to Spark.
for col_name in pandas_df.select_dtypes(include="object").columns:
    pandas_df[col_name] = pandas_df[col_name].astype(str)

# Convert the pandas DataFrame to a Spark DataFrame.
# From this point on, all processing is distributed across the cluster.
raw_df = spark.createDataFrame(pandas_df)

print(f"Done. {raw_df.count():,} rows loaded.")
raw_df.printSchema()
display(raw_df.limit(10))


---
## Part 2 — Curated Zone (Silver Layer)

The curated zone contains cleaned, validated data that is ready for analysis. The transformation from raw to curated is where most data engineering work happens.

We apply three transformations here:

1. **Parse the date column.** The xlsx format preserves datetime types, so a simple cast to `timestamp` is sufficient. If you loaded from CSV you would need `to_timestamp()` with an explicit format string — one of many reasons to prefer structured file formats over CSV.

2. **Remove rows with no customer ID.** About 135,000 transactions (25% of the dataset) have no `CustomerID`. These are guest checkouts. We cannot attribute them to a customer, so they are excluded from customer-level analysis. Note that after the `astype(str)` conversion earlier, missing values appear as the string `'nan'` rather than `NULL` — so we filter on both.

3. **Standardise column names.** Column names with capital letters and mixed casing cause friction in SQL. We rename to lowercase with underscores — the convention used across the rest of the lab.

In [0]:
from pyspark.sql.functions import col

clean_df = (
    raw_df
    # Cast InvoiceDate to a proper timestamp type
    .withColumn("InvoiceDate", col("InvoiceDate").cast("timestamp"))
    # Remove guest checkouts — 'nan' is how missing values appear after astype(str)
    .filter(col("CustomerID") != "nan")
    .filter(col("CustomerID").isNotNull())
)

# Rename columns to lowercase with underscores (SQL convention)
clean_df = (
    clean_df
    .withColumnRenamed("CustomerID", "customer_id")
    .withColumnRenamed("UnitPrice",  "unit_price")
    .withColumnRenamed("Quantity",   "quantity")
)

print(f"Rows after cleaning: {clean_df.count():,}")
print(f"Rows removed (guest checkouts): {raw_df.count() - clean_df.count():,}")
display(clean_df.limit(10))


---
## Part 3 — Writing to Delta Lake

Delta Lake is a storage layer that sits on top of your cloud file system and adds database-like features: ACID transactions, schema enforcement, and version history. It is what separates a proper data lake from a folder of CSV files.

We first drop the table if it already exists. This prevents a schema mismatch error if you re-run the notebook — Databricks persists tables across sessions, so a leftover table from a previous run would conflict with the new write. In production you would handle schema evolution more carefully, but for a lab environment a clean drop-and-recreate is the right approach.

After writing, we run a `COUNT(*)` to confirm the row count matches what we expect.

In [0]:
# Drop the table if it exists from a previous run.
# This avoids schema mismatch errors when re-running the notebook.
spark.sql("DROP TABLE IF EXISTS ecommerce_delta_table")

# Write the cleaned DataFrame as a managed Delta table.
# 'managed' means Databricks controls both the metadata and the data files.
(
    clean_df.write
    .format("delta")       # Delta Lake format — adds ACID + versioning
    .mode("overwrite")     # Replace any existing data
    .saveAsTable("ecommerce_delta_table")
)

print("Table written successfully.")
display(spark.sql("SELECT COUNT(*) AS row_count FROM ecommerce_delta_table"))


---
## Part 4 — Querying with SQL (Gold Layer)

Once data is in Delta Lake, you can query it with standard SQL — the same syntax you already know, now running on a distributed engine across the full dataset.

The query below computes total spending per customer and returns the top 10. A few things worth noticing:

- `quantity > 0` excludes returns (negative quantities). Returns are a legitimate part of the data but distort spending totals.
- `customer_id != 'nan'` is the same null guard from the cleaning step, applied here as a safety net.

After running the query, try clicking the **chart icon** below the results and switching to a bar chart to visualise the top customers.

In [0]:
%sql
-- Top 10 customers by total spending
SELECT
    customer_id,
    SUM(quantity)                        AS total_items_purchased,
    ROUND(SUM(unit_price * quantity), 2) AS total_spent
FROM ecommerce_delta_table
WHERE customer_id != 'nan'
  AND quantity > 0
GROUP BY customer_id
ORDER BY total_spent DESC
LIMIT 10;


---
## Part 5 — Time Travel

Every write operation on a Delta table creates a new version. This means you can query any previous state of the data — a feature called time travel. It is one of the properties that makes Delta Lake significantly more useful than plain Parquet or CSV storage.

Practical uses: reproducing last month's report on the exact data that existed then, recovering from an accidental delete, auditing what changed between two versions.

We will simulate this by running a `DELETE` to remove all returns (rows where `quantity < 0`), which creates Version 1 of the table. We can then query Version 0 — the original — to compare row counts and confirm we can go back.

**Run the four cells below in order.**

**Step 9a** — Check the version history. Right now there is only Version 0.

In [0]:
%sql
DESCRIBE HISTORY ecommerce_delta_table;


**Step 9b** — Delete return transactions. This creates Version 1.

In [0]:
%sql
DELETE FROM ecommerce_delta_table WHERE quantity < 0;


**Step 9c** — The history now shows two versions. Confirm Version 1 was created.

In [0]:
%sql
DESCRIBE HISTORY ecommerce_delta_table;


**Step 9d** — Compare row counts across versions. Version 0 should have more rows than the current table because it still includes the returns.

In [0]:
%sql
SELECT 'current (v1)' AS version, COUNT(*) AS row_count
FROM ecommerce_delta_table
UNION ALL
SELECT 'original (v0)' AS version, COUNT(*) AS row_count
FROM ecommerce_delta_table VERSION AS OF 0;


---
## What You Built

In this session you implemented a complete, if compact, data lake pipeline:

| Layer | What you did |
|---|---|
| **Raw (Bronze)** | Ingested 541K rows from an external source into Spark |
| **Curated (Silver)** | Cleaned types, removed invalid rows, standardised schema |
| **Analytics (Gold)** | Ran SQL aggregations on the distributed dataset |
| **Governance** | Used Delta Lake versioning and time travel |

The same architectural pattern — bronze, silver, gold — is used at Netflix, Airbnb, and most large-scale data organisations. The tooling scales; the principles do not change.

---
*DEGO 2606 — Nova SBE MSc Business Analytics*